[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/monacofj/misda/blob/main/benchmarks/optimization.ipynb)

# MISDA — optimization benchmark

This notebook asks a downstream question that the controlled diagnostics do not answer:

> **How much does optimization quality change when the full objective set is replaced by the MISDA-reduced set?**

For each classical MOP, MISDA first discovers a preferred objective subset from sampled full-objective data. NSGA-III is then run twice under paired seeds and equal search budgets:

- **Full** optimizes all original objectives.
- **Reduced** optimizes only the objectives retained by the preferred MIS.

The reduced search never sees the omitted objectives. For evaluation only, every decision vector produced by both searches is re-evaluated with the original MOP, so all final metrics and convergence histories are measured in the same original objective space.

The theoretical Pareto front is a common measuring reference for both treatments, not a third competitor.

In [ ]:
from pathlib import Path
import subprocess
import sys

# In a repository checkout, install the local package plus benchmark dependencies.
# In Colab, install MISDA from main. MoeaBench is pulled only through [benchmarks].
repo_root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists()), None)
REMOTE_REF = "main"
target = (
    f"{repo_root}[benchmarks]"
    if repo_root is not None
    else f"misda[benchmarks] @ git+https://github.com/monacofj/misda.git@{REMOTE_REF}"
)
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", target
])

In [ ]:
from types import SimpleNamespace

from IPython.display import display
import numpy as np
import pandas as pd
from pymoo.util.nds.non_dominated_sorting import NonDominatedSorting

import misda
import moeabench as mb

# Canonical M=3 is used first because MoeaBench's clinical/Q-score calibration
# resources are available for this dimension across the selected battery.
M = 3
DISCOVERY_N = 300
DISCOVERY_SEED = 123

POPULATION = 100
GENERATIONS = 200
REPEATS = 5
MOEA_SEED = 123
GT_POINTS = 5000

PROBLEM_FACTORIES = {
    "DTLZ2": lambda: mb.mops.DTLZ2(M=M),
    "DTLZ5": lambda: mb.mops.DTLZ5(M=M),
    "DTLZ7": lambda: mb.mops.DTLZ7(M=M),
    "DPF1": lambda: mb.mops.DPF1(M=M, D=2),
    "DPF3": lambda: mb.mops.DPF3(M=M, D=2),
    "DPF5": lambda: mb.mops.DPF5(M=M, D=2),
}

## Experimental protocol

The experiment deliberately separates **search space seen by the MOEA** from **space used to judge the result**.

1. Uniformly sample the original decision space and evaluate all original objectives.
2. Run MISDA on that full objective matrix and select the preferred MIS under the canonical ranking.
3. Construct a thin reduced MOP that preserves the original decision space but exposes only the selected objectives.
4. For each repetition, generate one initial decision population and inject that same `X0` into Full and Reduced NSGA-III runs; population size, generation budget, and run seed are also paired.
5. Re-evaluate every generation of both experiments with the original MOP.
6. Compute GD+, IGD+, and Hypervolume against the same original-space theoretical reference.
7. Compare final quality, topology, Q-Scores, and convergence by generation.

**Important:** re-evaluation is measurement instrumentation only. It never feeds omitted objectives back into the Reduced search.

The benchmark does not currently report an *active decision dimension* for these classical MOPs because MoeaBench does not expose an exact objective-to-decision-variable dependency declaration. No numerical sensitivity proxy is treated as ground truth.

In [ ]:
NDS = NonDominatedSorting()


class ReducedMop(mb.mops.BaseMop):
    """Original decision space with only a selected subset of objectives exposed."""

    def __init__(self, base_mop, selected_indices):
        self.base_mop = base_mop
        self.selected_indices = tuple(int(i) for i in selected_indices)
        if len(self.selected_indices) < 2:
            raise ValueError(
                "NSGA-III requires at least two retained objectives; "
                f"MISDA selected {len(self.selected_indices)}."
            )
        super().__init__(
            name=f"{base_mop.name}[MISDA]",
            M=len(self.selected_indices),
            N=base_mop.N,
            xl=np.asarray(base_mop.xl, dtype=float),
            xu=np.asarray(base_mop.xu, dtype=float),
        )

    def evaluation(self, X, n_ieq_constr=0):
        original = self.base_mop.evaluation(X, n_ieq_constr)
        reduced = dict(original)
        reduced["F"] = np.asarray(original["F"])[:, self.selected_indices]
        return reduced

    def ps(self, n_points=100):
        return self.base_mop.ps(n_points)


class FrontHistory:
    """Minimal run-like container accepted by MoeaBench metric evaluators."""

    def __init__(self, fronts, name):
        self.fronts = [np.asarray(front, dtype=float) for front in fronts]
        self.name = name

    def history(self, type="nd"):
        if type != "nd":
            raise ValueError("FrontHistory stores only non-dominated objective fronts")
        return self.fronts

    def front(self):
        return self.fronts[-1]


def nondominated(F):
    F = np.atleast_2d(np.asarray(F, dtype=float))
    if len(F) == 0:
        return F
    indices = NDS.do(F, only_non_dominated_front=True)
    return F[np.asarray(indices, dtype=int)]


def sample_decisions(mop, n, seed):
    rng = np.random.default_rng(seed)
    xl = np.broadcast_to(np.asarray(mop.xl, dtype=float), (mop.N,))
    xu = np.broadcast_to(np.asarray(mop.xu, dtype=float), (mop.N,))
    return rng.uniform(xl, xu, size=(int(n), mop.N))


def discover_reduction(mop):
    X = sample_decisions(mop, DISCOVERY_N, DISCOVERY_SEED)
    F = np.asarray(mop.evaluation(X)["F"], dtype=float)
    frame = pd.DataFrame(F, columns=[f"f{i}" for i in range(1, mop.M + 1)])

    mis_set = misda.discover(
        frame,
        name=f"{mop.name} optimization discovery",
        seed=DISCOVERY_SEED,
    )
    ranking = misda.rank(mis_set)
    selected = ranking.mis()
    return {
        "X": X,
        "F": F,
        "mis_set": mis_set,
        "ranking": ranking,
        "selected": selected,
        "selected_indices": tuple(selected.indices),
        "selected_objectives": tuple(str(x) for x in selected.objectives),
    }


def make_paired_experiments(original_mop, reduced_mop, problem_name):
    """Run Full and Reduced from identical X0 for every paired repetition."""
    full = mb.experiment()
    full.name = f"{problem_name} Full"
    full.mop = original_mop

    reduced = mb.experiment()
    reduced.name = f"{problem_name} Reduced"
    reduced.mop = reduced_mop

    for run_index in range(REPEATS):
        seed = MOEA_SEED + run_index
        X0 = sample_decisions(original_mop, POPULATION, seed)

        # Keep the MOEA run seed paired while reference directions retain their
        # dimension-specific, independently derived MoeaBench seed.
        full.moea = mb.moeas.NSGA3(
            population=POPULATION,
            generations=GENERATIONS,
            seed=MOEA_SEED,
            sampling=X0.copy(),
        )
        reduced.moea = mb.moeas.NSGA3(
            population=POPULATION,
            generations=GENERATIONS,
            seed=MOEA_SEED,
            sampling=X0.copy(),
        )

        full.run(repeat=1, append=run_index > 0, silent=True)
        reduced.run(repeat=1, append=run_index > 0, silent=True)

    assert_paired_initial_decisions(full, reduced)
    return full, reduced


def _sorted_rows(X):
    X = np.asarray(X, dtype=float)
    if len(X) == 0:
        return X
    order = np.lexsort(X.T[::-1])
    return X[order]


def assert_paired_initial_decisions(full, reduced):
    """Require the same generation-0 decision set for each paired seed."""
    if full.seed != reduced.seed:
        raise RuntimeError("Full and Reduced runs did not receive paired seeds")
    for full_run, reduced_run in zip(full.runs, reduced.runs):
        full_x0 = np.asarray(full_run.history("x")[0], dtype=float)
        reduced_x0 = np.asarray(reduced_run.history("x")[0], dtype=float)

        # Objective-dependent survival may reorder the same initial points.
        same_set = (
            full_x0.shape == reduced_x0.shape
            and np.allclose(_sorted_rows(full_x0), _sorted_rows(reduced_x0))
        )
        if not same_set:
            raise RuntimeError(
                f"Paired seed {full_run.seed} produced different initial decision sets"
            )


def full_space_histories(exp, original_mop):
    """Re-evaluate every stored decision population in the original objective space."""
    histories = []
    initial_populations = []
    for run in exp.runs:
        run_history = []
        decision_history = run.history("x")
        for Xg in decision_history:
            F_full = np.asarray(original_mop.evaluation(np.asarray(Xg))["F"], dtype=float)
            run_history.append(nondominated(F_full))
        histories.append(run_history)

        X0 = np.asarray(decision_history[0])
        initial_populations.append(
            np.asarray(original_mop.evaluation(X0)["F"], dtype=float)
        )
    return histories, initial_populations


def metric_history(histories, metric, reference, source_name, metric_name, **kwargs):
    run_like = [
        FrontHistory(fronts, f"{source_name} run {i + 1}")
        for i, fronts in enumerate(histories)
    ]
    matrix = metric(run_like, ref=reference, progress=False, **kwargs)
    return mb.metrics.MetricMatrix(
        matrix.values,
        metric_name=metric_name,
        source_name=source_name,
    )


def final_metric_row(name, full, reduced):
    full_values = np.asarray(full.gen(-1), dtype=float)
    reduced_values = np.asarray(reduced.gen(-1), dtype=float)
    full_mean = float(np.nanmean(full_values))
    reduced_mean = float(np.nanmean(reduced_values))
    change = reduced_mean - full_mean
    relative = change / abs(full_mean) if full_mean != 0 else np.nan
    return {
        "Metric": name,
        "Full": full_mean,
        "Reduced": reduced_mean,
        "Change": change,
        "Relative change": relative,
        "Full std": float(np.nanstd(full_values)),
        "Reduced std": float(np.nanstd(reduced_values)),
    }


def aggregate_final_front(histories):
    return nondominated(np.vstack([run_history[-1] for run_history in histories]))


def clinical_audits(histories, initial_populations, mop, reference):
    audits = []
    for run_history, initial in zip(histories, initial_populations):
        audit = mb.clinic.audit(
            run_history[-1],
            ground_truth=reference,
            problem=mop.name,
            k=POPULATION,
            initial_data=initial,
        )
        audits.append(audit)
    return audits


def mean_q_profile(audits, name):
    if not audits or any(audit.quality is None for audit in audits):
        return None
    keys = tuple(audits[0].quality.scores)
    scores = {
        key: float(np.mean([
            float(audit.quality.scores[key].value)
            for audit in audits
        ]))
        for key in keys
    }
    return SimpleNamespace(name=name, scores=scores)


def qscore_table(full_profile, reduced_profile):
    if full_profile is None or reduced_profile is None:
        return pd.DataFrame()
    rows = []
    for key in full_profile.scores:
        full = float(full_profile.scores[key])
        reduced = float(reduced_profile.scores[key])
        rows.append({
            "Q-Score": key,
            "Full": full,
            "Reduced": reduced,
            "Change": reduced - full,
        })
    return pd.DataFrame(rows)


def run_problem(problem_name, *, show=True):
    mop = PROBLEM_FACTORIES[problem_name]()
    discovery = discover_reduction(mop)

    print(f"{problem_name}: MISDA objectives {mop.M} → {len(discovery['selected_indices'])}")
    print("Selected:", ", ".join(discovery["selected_objectives"]))

    if len(discovery["selected_indices"]) < 2:
        print("Optimization comparison skipped: NSGA-III needs at least two objectives.")
        return {
            "problem": problem_name,
            "mop": mop,
            "discovery": discovery,
            "status": "skipped_single_objective",
        }

    reduced_mop = ReducedMop(mop, discovery["selected_indices"])

    full, reduced = make_paired_experiments(mop, reduced_mop, problem_name)

    full_histories, full_initial = full_space_histories(full, mop)
    reduced_histories, reduced_initial = full_space_histories(reduced, mop)

    reference = np.asarray(mop.pf(GT_POINTS), dtype=float)
    reference = nondominated(reference)

    gd_full = metric_history(
        full_histories, mb.metrics.gdplus, reference, "Full", "GD+"
    )
    gd_reduced = metric_history(
        reduced_histories, mb.metrics.gdplus, reference, "Reduced", "GD+"
    )
    igd_full = metric_history(
        full_histories, mb.metrics.igdplus, reference, "Full", "IGD+"
    )
    igd_reduced = metric_history(
        reduced_histories, mb.metrics.igdplus, reference, "Reduced", "IGD+"
    )
    hv_full = metric_history(
        full_histories,
        mb.metrics.hv,
        reference,
        "Full",
        "Hypervolume",
        mode="exact",
        scale="raw",
    )
    hv_reduced = metric_history(
        reduced_histories,
        mb.metrics.hv,
        reference,
        "Reduced",
        "Hypervolume",
        mode="exact",
        scale="raw",
    )

    performance = pd.DataFrame([
        final_metric_row("GD+", gd_full, gd_reduced),
        final_metric_row("IGD+", igd_full, igd_reduced),
        final_metric_row("HV", hv_full, hv_reduced),
    ])

    full_audits = clinical_audits(full_histories, full_initial, mop, reference)
    reduced_audits = clinical_audits(reduced_histories, reduced_initial, mop, reference)
    q_full = mean_q_profile(full_audits, "Full")
    q_reduced = mean_q_profile(reduced_audits, "Reduced")
    qscores = qscore_table(q_full, q_reduced)

    print(
        f"Budget per treatment: population={POPULATION}, generations={GENERATIONS}, "
        f"repeats={REPEATS}, evaluations/run≈{POPULATION * GENERATIONS}"
    )
    display(performance)

    full_final = aggregate_final_front(full_histories)
    reduced_final = aggregate_final_front(reduced_histories)

    if show:
        mb.view.topology(
            full_final,
            reduced_final,
            gt=reference,
            labels=["Full", "Reduced"],
            title=f"{problem_name} — full-space final fronts",
        )

        if q_full is not None and q_reduced is not None:
            display(qscores)
            mb.view.radar(
                q_full,
                q_reduced,
                title=f"{problem_name} — mean clinical Q-Scores",
            )
        else:
            print("Q-Scores unavailable: no compatible MoeaBench clinical baseline.")

        mb.view.history(
            igd_full,
            igd_reduced,
            domain="perf",
            labels=["Full", "Reduced"],
            title=f"{problem_name} — IGD+ convergence",
        )
        mb.view.history(
            hv_full,
            hv_reduced,
            domain="perf",
            labels=["Full", "Reduced"],
            title=f"{problem_name} — Hypervolume convergence",
        )

    return {
        "problem": problem_name,
        "mop": mop,
        "reduced_mop": reduced_mop,
        "discovery": discovery,
        "full": full,
        "reduced": reduced,
        "reference": reference,
        "full_histories": full_histories,
        "reduced_histories": reduced_histories,
        "metrics": {
            "gdplus": (gd_full, gd_reduced),
            "igdplus": (igd_full, igd_reduced),
            "hv": (hv_full, hv_reduced),
        },
        "performance": performance,
        "q_full": q_full,
        "q_reduced": q_reduced,
        "qscores": qscores,
        "status": "completed",
    }

## Reading the results

For GD+ and IGD+, smaller values are better; for Hypervolume, larger values are better. The notebook therefore reports the signed numerical change **Reduced − Full** without turning it into a winner or score.

The topology view is explanatory only. The numerical metrics always use the complete original objective space.

The Q-Score radar is also comparative: each profile is the mean of the standard six MoeaBench Q-Scores over paired runs. Headway remains in the standard radar.

## DTLZ2

Regular spherical baseline: a control case where strong reduction is not expected.

In [ ]:
dtlz2_result = run_problem("DTLZ2")

## DTLZ5

Classical degenerate spherical problem.

In [ ]:
dtlz5_result = run_problem("DTLZ5")

## DTLZ7

Disconnected Pareto-front problem.

In [ ]:
dtlz7_result = run_problem("DTLZ7")

## DPF1

Degenerate projection problem with explicit redundant objectives.

In [ ]:
dpf1_result = run_problem("DPF1")

## DPF3

Nonlinear chaos/min-max projection problem.

In [ ]:
dpf3_result = run_problem("DPF3")

## DPF5

Conditional degenerate problem whose objective mapping changes with the decision regime.

In [ ]:
dpf5_result = run_problem("DPF5")

# Cross-problem summary

The compact table below answers the benchmark question across the complete battery: how much objective reduction MISDA obtained and how the final optimization quality changed under the same search budget.

In [ ]:
optimization_results = {
    name: globals()[f"{name.lower()}_result"]
    for name in PROBLEM_FACTORIES
}

summary_rows = []
for name, result in optimization_results.items():
    selected_dimension = len(result["discovery"]["selected_indices"])
    row = {
        "Problem": name,
        "Original objectives": result["mop"].M,
        "Selected objectives": selected_dimension,
        "Reduction fraction": 1.0 - selected_dimension / result["mop"].M,
        "Status": result["status"],
    }
    if result["status"] == "completed":
        for _, metric_row in result["performance"].iterrows():
            metric = metric_row["Metric"]
            row[f"{metric} Full"] = metric_row["Full"]
            row[f"{metric} Reduced"] = metric_row["Reduced"]
            row[f"{metric} Change"] = metric_row["Change"]
    summary_rows.append(row)

optimization_summary = pd.DataFrame(summary_rows)
optimization_summary